<a href="https://colab.research.google.com/github/GabooVZ29/Sports-Analytics-Series/blob/main/Proyecto_MLB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install plotly -q

import csv
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Librerías listas")

✅ Librerías listas


In [31]:
#Carga y Limpieza de data

# --- Salarios históricos (1988-2010) ---
rows = []
with open('salaries.csv', 'r', encoding='ascii') as f:
    reader = csv.reader(f)
    headers = next(reader)
    for row in reader:
        if len(row) == len(headers):
            rows.append(row)

sal_old = pd.DataFrame(rows, columns=headers)
sal_old['year'] = sal_old['years'].str.extract(r'\((\d{4})\)').astype(float).astype('Int64')
sal_old['salary_val'] = sal_old['salary'].str.replace(r'[\$,\s]', '', regex=True).astype(float)

# --- Salarios modernos (2011-2024) ---
sal_modern = pd.read_csv('mlb_salary_data.csv', encoding='ISO-8859-1')
sal_modern['salary_val'] = sal_modern['Salary'].str.replace(r'[\$,]', '', regex=True).astype(float)
sal_modern.rename(columns={'Year': 'year', 'Team': 'team'}, inplace=True)

# --- Resultados por equipo ---
teams = pd.read_csv('Teams (1).csv', encoding='utf-8-sig')

print("✅ Datos cargados")
print(f"   Salarios históricos: {len(sal_old)} filas")
print(f"   Salarios modernos: {len(sal_modern)} filas")
print(f"   Resultados equipos: {len(teams)} filas")

✅ Datos cargados
   Salarios históricos: 22836 filas
   Salarios modernos: 13954 filas
   Resultados equipos: 3614 filas


In [32]:
# Mapeo de abrevaciones entre archivos

team_map = {'NYY': 'NYA', 'TB': 'TBA', 'TBD': 'TBA', 'LAD': 'LAN', 'SF': 'SFN'}

# Agregar payroll por equipo/año
payroll_old = sal_old[sal_old['year'].between(1995, 2010)].groupby(['year','team'])['salary_val'].sum().reset_index()
payroll_new = sal_modern[sal_modern['year'].between(2011, 2024)].groupby(['year','team'])['salary_val'].sum().reset_index()
payroll = pd.concat([payroll_old, payroll_new])
payroll['team_id'] = payroll['team'].map(team_map).fillna(payroll['team'])

# Cruzar con resultados
teams_sub = teams[['yearID','teamID','W','L','DivWin','WCWin','LgWin','WSWin']].copy()
teams_sub.rename(columns={'yearID':'year','teamID':'team_id'}, inplace=True)

merged = payroll.merge(teams_sub, on=['year','team_id'], how='inner')
merged['win_pct'] = merged['W'] / (merged['W'] + merged['L'])
merged['payroll_M'] = merged['salary_val'] / 1_000_000

# Ratio vs promedio de la liga
lg_avg = merged.groupby('year')['payroll_M'].mean().reset_index(name='lg_avg')
merged = merged.merge(lg_avg, on='year')
merged['payroll_ratio'] = merged['payroll_M'] / merged['lg_avg']

# Etiquetar equipos y eras
def label_team(row):
    if row['team_id'] == 'NYA' and row['year'] <= 2005:
        return "Yankees 1995-2005"
    elif row['team_id'] == 'NYA' and row['year'] > 2005:
        return "Yankees 2006-2024"
    elif row['team_id'] == 'TBA':
        return "Rays 2008-2024"
    elif row['team_id'] == 'LAN':
        return "Dodgers 2010-2024"
    elif row['team_id'] == 'SFN' and (row['year'] >= 2010 and row['year'] <= 2014):
        return "Giants 2010-2014"
    return None

merged['team_label'] = merged.apply(label_team, axis=1)
df = merged[merged['team_label'].notna()].copy()

# Marcar logros
df['resultado'] = '● Regular'
df.loc[df['LgWin'] == 'Y', 'resultado'] = '◎ Pennant'
df.loc[df['WSWin'] == 'Y', 'resultado'] = '▲ WS Champion'

print("✅ Dataset listo:", len(df), "temporadas")
print(df.groupby('team_label')[['payroll_M','W','win_pct']].mean().round(2))

✅ Dataset listo: 92 temporadas
                   payroll_M      W  win_pct
team_label                                  
Dodgers 2010-2024     130.42  88.50     0.56
Giants 2010-2014      123.93  87.20     0.54
Rays 2008-2024         49.76  77.44     0.49
Yankees 1995-2005     107.79  96.45     0.60
Yankees 2006-2024     197.35  89.68     0.57


In [33]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# 1. Cálculo rápido del R2 para mostrarlo en el gráfico
model = LinearRegression()
model.fit(df[['payroll_M']], df['win_pct'])
r2_value = model.score(df[['payroll_M']], df['win_pct'])

# 2. Crear el gráfico interactivo
fig = px.scatter(df,
                 x='payroll_M',
                 y='win_pct',
                 hover_name=df.index,
                 trendline='ols',
                 title=f'¿EL DINERO COMPRA VICTORIAS? (R² = {r2_value:.2f})',
                 labels={'payroll_M': 'Payroll (USD Millions)', 'win_pct': 'Win Percentage'},
                 template='plotly_dark')

# 3. Estilo de los puntos
fig.update_traces(marker=dict(size=12, color='#00d4ff', opacity=0.7,
                              line=dict(width=1, color='White')),
                  selector=dict(mode='markers'))

# 4. Estilo de la línea de regresion lineal
fig.update_traces(line=dict(color='#ff0055', width=4),
                  selector=dict(type='scatter', mode='lines'))

# 5. Ajuste del fondo
fig.update_layout(
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font_color="white",
    title_font_size=20,
    xaxis=dict(showgrid=True, gridcolor='#444444', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#444444', zeroline=False),
    margin=dict(l=40, r=40, t=80, b=40)
)

fig.show()

In [34]:
COLORS = {
    "Yankees 1995-2005": "#C4A962",
    "Yankees 2006-2024": "#7A8FA6",
    "Rays 2008-2024":    "#38B2AC",
    "Dodgers 2010-2024": "#3B82F6",
    "Giants 2010-2014":  "#F97316",
}

# --- Gráfico 1: Scatter Payroll vs Win% ---
fig1 = px.scatter(
    df, x='payroll_M', y='win_pct',
    color='team_label', symbol='resultado',
    hover_data=['year', 'W', 'payroll_ratio'],
    color_discrete_map=COLORS,
    title='Payroll vs Win% — cada punto es una temporada',
    labels={'payroll_M': 'Payroll ($M)', 'win_pct': 'Win %', 'team_label': 'Equipo'}
)
fig1.add_hline(y=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig1.update_layout(template='plotly_dark', height=500)
fig1.show()

# --- Gráfico 2: Ratio vs liga a lo largo del tiempo ---
fig2 = px.line(
    df, x='year', y='payroll_ratio',
    color='team_label', markers=True,
    color_discrete_map=COLORS,
    title='Ratio Payroll vs Promedio de la Liga por año',
    labels={'payroll_ratio': 'Ratio vs Liga', 'year': 'Año', 'team_label': 'Equipo'}
)
fig2.add_hline(y=1.0, line_dash='dash', line_color='#C4A962', opacity=0.5,
               annotation_text='Promedio MLB')
fig2.update_layout(template='plotly_dark', height=450)
fig2.show()

# --- Gráfico 3: Eficiencia WS ---
summary = df.groupby('team_label').agg(
    ws=('WSWin', lambda x: (x=='Y').sum()),
    avg_ratio=('payroll_ratio', 'mean'),
    seasons=('year', 'count'),
    avg_W=('W', 'mean')
).reset_index()
summary['eficiencia'] = summary['ws'] / (summary['avg_ratio'] * summary['seasons']) * 100

fig3 = px.bar(
    summary.sort_values('eficiencia', ascending=True),
    x='eficiencia', y='team_label', orientation='h',
    color='team_label', color_discrete_map=COLORS,
    title='Eficiencia WS (títulos por unidad de gasto relativo)',
    text='ws',
    labels={'eficiencia': 'Eficiencia', 'team_label': 'Equipo'}
)
fig3.update_traces(texttemplate='%{text} WS', textposition='outside')
fig3.update_layout(template='plotly_dark', height=380, showlegend=False)
fig3.show()

print("\n📊 Resumen final:")
print(summary[['team_label','avg_ratio','avg_W','ws','eficiencia']].round(3).to_string(index=False))


📊 Resumen final:
       team_label  avg_ratio  avg_W  ws  eficiencia
Dodgers 2010-2024      1.452 88.500   2       4.591
 Giants 2010-2014      1.231 87.200   3      48.736
   Rays 2008-2024      0.556 77.444   0         0.0
Yankees 1995-2005      1.829 96.455   4      19.882
Yankees 2006-2024      1.928 89.684   1        2.73


In [35]:
df.to_csv('mlb_final.csv', index=False)